# UMTAM Task Interference Analysis (Memory-Optimized)

**Objective:** Demonstrate that UMTAM's trajectory-informed merging provides greater benefits in scenarios with high parameter conflict (Reviewer Comment #9)

**Memory Optimization:** This version saves task experts to disk and processes pairs sequentially to fit within T4 Colab's 15GB RAM limit.

**Expected Runtime:** ~1.5-2 hours on T4 GPU

---
## 📦 Setup

In [ ]:
# Install packages
!pip install -q transformers datasets torch evaluate matplotlib seaborn scipy
print("✓ Packages installed")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import copy
import warnings
import gc
import os
from itertools import combinations
from scipy import stats
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create directory for saving experts
os.makedirs('experts', exist_ok=True)

print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Check RAM
import psutil
print(f"✓ System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"✓ Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")

In [ ]:
def print_memory_usage():
    """Print current memory usage"""
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated() / 1e9
        gpu_cached = torch.cuda.memory_reserved() / 1e9
        print(f"  GPU: {gpu_mem:.2f} GB allocated, {gpu_cached:.2f} GB cached")
    ram = psutil.virtual_memory()
    print(f"  RAM: {ram.used / 1e9:.2f} GB used / {ram.total / 1e9:.1f} GB total ({ram.percent}%)")

def clear_memory():
    """Aggressively clear memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print("Initial memory state:")
print_memory_usage()

---
## ⚙️ Configuration

In [ ]:
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    
    # Tasks - 4 GLUE tasks giving us 6 unique pairs
    'tasks': {
        'rte': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'rte',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'mrpc': {
            'num_labels': 2,
            'metric': 'f1',
            'glue_name': 'mrpc',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'qnli': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'qnli',
            'text_fields': ('question', 'sentence'),
        },
        'cola': {
            'num_labels': 2,
            'metric': 'matthews_correlation',
            'glue_name': 'cola',
            'text_fields': ('sentence', None),
        },
    },
    
    # Training - reduced for memory efficiency
    'max_length': 128,
    'batch_size': 16,  # Reduced from 32
    'num_epochs': 3,
    'lr': 2e-5,
    
    # Merging
    'sparsity_k': 20,
    
    # Other
    'seed': 42,
    'max_train_samples': 2500,  # Reduced for memory
    'max_eval_samples': 400,
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print("✓ Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Tasks: {list(CONFIG['tasks'].keys())}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Max train samples: {CONFIG['max_train_samples']}")

---
## 📚 Load Datasets (One at a Time)

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def load_dataset_for_task(task_name, task_config):
    """Load and tokenize a single GLUE dataset - call only when needed"""
    dataset = load_dataset('glue', task_config['glue_name'])
    
    text1, text2 = task_config['text_fields']
    
    def tokenize_fn(examples):
        if text2 is None:
            result = tokenizer(
                examples[text1],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        else:
            result = tokenizer(
                examples[text1],
                examples[text2],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        result['labels'] = examples['label']
        return result
    
    tokenized = dataset.map(tokenize_fn, batched=True)
    
    train_data = tokenized['train'].shuffle(seed=CONFIG['seed'])
    if CONFIG['max_train_samples'] and len(train_data) > CONFIG['max_train_samples']:
        train_data = train_data.select(range(CONFIG['max_train_samples']))
    
    eval_data = tokenized['validation']
    if CONFIG['max_eval_samples'] and len(eval_data) > CONFIG['max_eval_samples']:
        eval_data = eval_data.select(range(CONFIG['max_eval_samples']))
    
    train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    return train_data, eval_data

print("✓ Dataset loader defined (datasets loaded on-demand)")

---
## 🏋️ Training with Disk-Based Storage

In [ ]:
def train_and_save_expert(task_name, task_config, base_state_dict):
    """
    Train a task expert and SAVE TO DISK immediately.
    Only keeps minimal data in memory.
    """
    print(f"\n{'='*60}")
    print(f"Training {task_name.upper()} Expert")
    print(f"{'='*60}")
    print_memory_usage()
    
    # Load dataset for this task only
    train_data, eval_data = load_dataset_for_task(task_name, task_config)
    print(f"  Train: {len(train_data)}, Eval: {len(eval_data)}")
    
    # Create model
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    
    # Load base weights for encoder
    model_dict = model.state_dict()
    for name, param in base_state_dict.items():
        if name in model_dict and 'classifier' not in name:
            model_dict[name] = param.clone()
    model.load_state_dict(model_dict)
    model.to(device)
    model.train()
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    train_loader = DataLoader(train_data, batch_size=CONFIG['batch_size'], shuffle=True)
    
    # Get layer names for weight matrices (encoder only)
    layer_names = [name for name, param in model.named_parameters() 
                   if len(param.shape) == 2 and 'classifier' not in name]
    
    # Initialize curvature accumulators ON GPU
    curvature = {}
    for name in layer_names:
        param = dict(model.named_parameters())[name]
        m, n = param.shape
        curvature[name] = {
            'R': torch.zeros(m, device=device),
            'C': torch.zeros(n, device=device),
        }
    
    num_steps = 0
    
    # Training loop
    for epoch in range(CONFIG['num_epochs']):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
        
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            
            # Accumulate curvature
            with torch.no_grad():
                for name in layer_names:
                    param = dict(model.named_parameters())[name]
                    if param.grad is not None:
                        grad_sq = param.grad ** 2
                        curvature[name]['R'] += grad_sq.sum(dim=1)
                        curvature[name]['C'] += grad_sq.sum(dim=0)
            
            num_steps += 1
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        print(f"  Epoch {epoch+1} loss: {total_loss / len(train_loader):.4f}")
    
    # Normalize curvature
    for name in curvature:
        curvature[name]['R'] /= num_steps
        curvature[name]['C'] /= num_steps
    
    # Compute task vector and saliency - DIRECTLY SAVE TO DISK
    print("  Computing task vector and saliency...")
    
    expert_data = {
        'task_vector': {},
        'saliency': {},
        'curvature': {},
    }
    
    for name in layer_names:
        param = dict(model.named_parameters())[name]
        w0 = base_state_dict[name].to(device)
        
        # Task vector
        delta = param.data - w0
        expert_data['task_vector'][name] = delta.cpu()
        
        # Saliency = delta² × sqrt(R ⊗ C)
        R = curvature[name]['R'].clamp(min=1e-8)
        C = curvature[name]['C'].clamp(min=1e-8)
        curv_matrix = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        saliency = (delta ** 2) * curv_matrix
        expert_data['saliency'][name] = saliency.cpu()
        
        # Curvature (move to CPU)
        expert_data['curvature'][name] = {
            'R': curvature[name]['R'].cpu(),
            'C': curvature[name]['C'].cpu(),
        }
        
        # Clear GPU tensor
        del delta, curv_matrix, saliency
    
    # Save to disk
    save_path = f'experts/{task_name}_expert.pt'
    torch.save(expert_data, save_path)
    print(f"  ✓ Saved to {save_path}")
    
    # Also save eval dataset reference
    eval_path = f'experts/{task_name}_eval.pt'
    torch.save({
        'input_ids': eval_data['input_ids'],
        'attention_mask': eval_data['attention_mask'],
        'labels': eval_data['labels'],
    }, eval_path)
    
    # Save train dataset for classifier fine-tuning
    train_path = f'experts/{task_name}_train.pt'
    torch.save({
        'input_ids': train_data['input_ids'],
        'attention_mask': train_data['attention_mask'],
        'labels': train_data['labels'],
    }, train_path)
    
    # Cleanup
    del model, optimizer, train_loader, train_data, eval_data, curvature, expert_data
    clear_memory()
    
    print(f"✓ {task_name.upper()} expert trained and saved")
    print_memory_usage()
    
    return save_path

print("✓ Training function defined")

In [ ]:
def load_expert(task_name):
    """Load expert from disk"""
    return torch.load(f'experts/{task_name}_expert.pt')

def load_eval_data(task_name):
    """Load evaluation data from disk"""
    return torch.load(f'experts/{task_name}_eval.pt')

def load_train_data(task_name):
    """Load training data from disk"""
    return torch.load(f'experts/{task_name}_train.pt')

print("✓ Helper functions defined")

---
## 📊 Interference Metrics (Memory-Efficient)

In [ ]:
def compute_interference_metrics(task1_name, task2_name, k_percent=20):
    """
    Compute interference metrics by loading experts from disk.
    Processes layer by layer to minimize memory.
    """
    expert1 = load_expert(task1_name)
    expert2 = load_expert(task2_name)
    
    metrics = {
        'task_pair': f"{task1_name}-{task2_name}",
        'task1': task1_name,
        'task2': task2_name,
    }
    
    total_params = 0
    sign_conflicts = 0
    magnitude_conflict = 0.0
    saliency_conflict = 0.0
    total_magnitude = 0.0
    total_saliency = 0.0
    
    # For overlap calculation
    all_sal1 = []
    all_sal2 = []
    all_tv1 = []
    all_tv2 = []
    
    layer_names = list(expert1['task_vector'].keys())
    
    for name in layer_names:
        tv1 = expert1['task_vector'][name]
        tv2 = expert2['task_vector'][name]
        sal1 = expert1['saliency'][name]
        sal2 = expert2['saliency'][name]
        
        # Sign conflicts
        sign1 = torch.sign(tv1)
        sign2 = torch.sign(tv2)
        both_active = (tv1.abs() > 1e-8) & (tv2.abs() > 1e-8)
        conflict_mask = (sign1 != sign2) & both_active
        
        sign_conflicts += conflict_mask.sum().item()
        total_params += both_active.sum().item()
        
        # Magnitude-weighted conflict
        mag_product = tv1.abs() * tv2.abs()
        magnitude_conflict += (mag_product * conflict_mask.float()).sum().item()
        total_magnitude += mag_product.sum().item()
        
        # Saliency-weighted conflict
        sal_product = sal1 * sal2
        saliency_conflict += (sal_product * conflict_mask.float()).sum().item()
        total_saliency += sal_product.sum().item()
        
        # Collect for overlap (subsample to save memory)
        all_sal1.append(sal1.flatten())
        all_sal2.append(sal2.flatten())
        all_tv1.append(tv1.flatten())
        all_tv2.append(tv2.flatten())
    
    # Compute basic metrics
    metrics['sign_conflict_rate'] = sign_conflicts / max(total_params, 1)
    metrics['magnitude_conflict'] = magnitude_conflict / max(total_magnitude, 1e-8)
    metrics['saliency_conflict'] = saliency_conflict / max(total_saliency, 1e-8)
    
    # Concatenate for overlap calculation
    sal1_all = torch.cat(all_sal1)
    sal2_all = torch.cat(all_sal2)
    tv1_all = torch.cat(all_tv1)
    tv2_all = torch.cat(all_tv2)
    
    # Task vector distance
    metrics['task_vector_distance'] = torch.norm(tv1_all - tv2_all).item()
    
    # Cosine similarity
    cos_sim = torch.nn.functional.cosine_similarity(tv1_all.unsqueeze(0), tv2_all.unsqueeze(0))
    metrics['cosine_similarity'] = cos_sim.item()
    
    # Overlap conflict
    k = int(len(sal1_all) * k_percent / 100)
    _, topk1 = torch.topk(sal1_all, k)
    _, topk2 = torch.topk(sal2_all, k)
    
    overlap = set(topk1.tolist()) & set(topk2.tolist())
    if len(overlap) > 0:
        overlap_idx = list(overlap)
        signs1 = torch.sign(tv1_all[overlap_idx])
        signs2 = torch.sign(tv2_all[overlap_idx])
        metrics['overlap_conflict'] = (signs1 != signs2).float().mean().item()
        metrics['overlap_size'] = len(overlap)
    else:
        metrics['overlap_conflict'] = 0.0
        metrics['overlap_size'] = 0
    
    # Cleanup
    del expert1, expert2, sal1_all, sal2_all, tv1_all, tv2_all
    clear_memory()
    
    return metrics

print("✓ Interference metrics function defined")

---
## 🔀 Merging Methods (Memory-Efficient)

In [ ]:
def merge_ties_pair(base_state, task1_name, task2_name, k_percent=20):
    """TIES-Merging - loads experts from disk"""
    expert1 = load_expert(task1_name)
    expert2 = load_expert(task2_name)
    
    merged_state = copy.deepcopy(base_state)
    layer_names = list(expert1['task_vector'].keys())
    
    for name in layer_names:
        tv1 = expert1['task_vector'][name]
        tv2 = expert2['task_vector'][name]
        
        # Trim by magnitude
        mag1 = tv1.abs().flatten()
        mag2 = tv2.abs().flatten()
        k = int(len(mag1) * k_percent / 100)
        
        thresh1 = torch.kthvalue(mag1, len(mag1) - k).values if k > 0 else mag1.max()
        thresh2 = torch.kthvalue(mag2, len(mag2) - k).values if k > 0 else mag2.max()
        
        mask1 = (tv1.abs() >= thresh1).float()
        mask2 = (tv2.abs() >= thresh2).float()
        
        trimmed1 = tv1 * mask1
        trimmed2 = tv2 * mask2
        
        # Sign election
        pos_mass = (trimmed1 > 0).float() * trimmed1.abs() + (trimmed2 > 0).float() * trimmed2.abs()
        neg_mass = (trimmed1 < 0).float() * trimmed1.abs() + (trimmed2 < 0).float() * trimmed2.abs()
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Disjoint merge
        merged_delta = torch.zeros_like(tv1)
        count = torch.zeros_like(tv1)
        
        for tv, mask in [(trimmed1, mask1), (trimmed2, mask2)]:
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            merged_delta += tv * agree_mask
            count += agree_mask
        
        merged_delta = torch.where(count > 0, merged_delta / count.clamp(min=1), merged_delta)
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    del expert1, expert2
    clear_memory()
    
    return merged_state


def merge_umtam_pair(base_state, task1_name, task2_name, k_percent=20):
    """UMTAM Curvature-Aware Merging - loads experts from disk"""
    expert1 = load_expert(task1_name)
    expert2 = load_expert(task2_name)
    
    merged_state = copy.deepcopy(base_state)
    layer_names = list(expert1['task_vector'].keys())
    
    for name in layer_names:
        tv1 = expert1['task_vector'][name]
        tv2 = expert2['task_vector'][name]
        sal1 = expert1['saliency'][name]
        sal2 = expert2['saliency'][name]
        
        # Normalize saliency
        sal1_norm = sal1 / (sal1.max() + 1e-8)
        sal2_norm = sal2 / (sal2.max() + 1e-8)
        
        # Trim by saliency
        flat_sal1 = sal1_norm.flatten()
        flat_sal2 = sal2_norm.flatten()
        k = int(len(flat_sal1) * k_percent / 100)
        
        thresh1 = torch.kthvalue(flat_sal1, len(flat_sal1) - k).values if k > 0 else flat_sal1.max()
        thresh2 = torch.kthvalue(flat_sal2, len(flat_sal2) - k).values if k > 0 else flat_sal2.max()
        
        mask1 = (sal1_norm >= thresh1).float()
        mask2 = (sal2_norm >= thresh2).float()
        
        # Sign election weighted by saliency
        pos_mass = (tv1 > 0).float() * sal1_norm * mask1 + (tv2 > 0).float() * sal2_norm * mask2
        neg_mass = (tv1 < 0).float() * sal1_norm * mask1 + (tv2 < 0).float() * sal2_norm * mask2
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Curvature-weighted merge
        weighted_sum = torch.zeros_like(tv1)
        weight_total = torch.zeros_like(tv1)
        
        for tv, mask, curv in [(tv1, mask1, expert1['curvature'][name]),
                                (tv2, mask2, expert2['curvature'][name])]:
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            
            R = curv['R'].clamp(min=1e-8)
            C = curv['C'].clamp(min=1e-8)
            curv_weight = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
            curv_weight = curv_weight / (curv_weight.max() + 1e-8)
            
            weighted_sum += tv * agree_mask * curv_weight
            weight_total += agree_mask * curv_weight
        
        merged_delta = torch.where(
            weight_total > 0,
            weighted_sum / weight_total.clamp(min=1e-8),
            torch.zeros_like(weighted_sum)
        )
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    del expert1, expert2
    clear_memory()
    
    return merged_state

print("✓ Merging methods defined")

---
## 📊 Evaluation (Memory-Efficient)

In [ ]:
def evaluate_merged_on_pair(merged_state, base_state, task1_name, task2_name):
    """Evaluate merged model on both tasks"""
    scores = {}
    
    for task_name in [task1_name, task2_name]:
        task_config = CONFIG['tasks'][task_name]
        
        # Load data from disk
        train_data = load_train_data(task_name)
        eval_data = load_eval_data(task_name)
        
        # Create model
        model = AutoModelForSequenceClassification.from_pretrained(
            CONFIG['model_name'],
            num_labels=task_config['num_labels']
        )
        
        # Load merged weights
        model_dict = model.state_dict()
        for name, param in merged_state.items():
            if name in model_dict and 'classifier' not in name:
                model_dict[name] = param.clone()
        model.load_state_dict(model_dict)
        model.to(device)
        
        # Quick classifier fine-tuning
        for name, param in model.named_parameters():
            param.requires_grad = 'classifier' in name
        
        optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        
        # Create simple dataloader from saved tensors
        train_dataset = torch.utils.data.TensorDataset(
            train_data['input_ids'],
            train_data['attention_mask'],
            train_data['labels']
        )
        train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
        
        model.train()
        for input_ids, attention_mask, labels in train_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        metric = evaluate.load('glue', task_config['glue_name'])
        
        eval_dataset = torch.utils.data.TensorDataset(
            eval_data['input_ids'],
            eval_data['attention_mask'],
            eval_data['labels']
        )
        eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size'])
        
        with torch.no_grad():
            for input_ids, attention_mask, labels in eval_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                predictions = torch.argmax(outputs.logits, dim=-1)
                metric.add_batch(predictions=predictions.cpu(), references=labels)
        
        result = metric.compute()
        scores[task_name] = result[task_config['metric']]
        
        del model, optimizer, train_loader, eval_loader, train_data, eval_data
        clear_memory()
    
    scores['average'] = np.mean(list(scores.values()))
    return scores

print("✓ Evaluation function defined")

---
## 🚀 Run Experiment

In [ ]:
print("="*70)
print("TASK INTERFERENCE ANALYSIS EXPERIMENT")
print("="*70)

# Step 1: Save base model state
print("\n[Step 1] Loading base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=2
)
base_state_dict = {k: v.cpu().clone() for k, v in base_model.state_dict().items()}

# Save base state to disk
torch.save(base_state_dict, 'experts/base_state.pt')

del base_model
clear_memory()
print("✓ Base model state saved to disk")
print_memory_usage()

In [ ]:
# Step 2: Train all experts (one at a time, save to disk)
print("\n[Step 2] Training task experts...")

for task_name, task_config in CONFIG['tasks'].items():
    # Load base state fresh each time
    base_state = torch.load('experts/base_state.pt')
    
    train_and_save_expert(task_name, task_config, base_state)
    
    del base_state
    clear_memory()
    print(f"\nAfter {task_name}:")
    print_memory_usage()

print(f"\n✓ All {len(CONFIG['tasks'])} experts trained and saved to disk")

In [ ]:
# Step 3: Compute interference metrics for all pairs
print("\n[Step 3] Computing interference metrics...")
print("="*70)

task_names = list(CONFIG['tasks'].keys())
task_pairs = list(combinations(task_names, 2))

interference_data = []

for task1, task2 in task_pairs:
    metrics = compute_interference_metrics(task1, task2, k_percent=CONFIG['sparsity_k'])
    interference_data.append(metrics)
    
    print(f"\n{task1.upper()}-{task2.upper()}:")
    print(f"  Sign conflict rate: {metrics['sign_conflict_rate']:.3f}")
    print(f"  Saliency conflict:  {metrics['saliency_conflict']:.3f}")
    print(f"  Overlap conflict:   {metrics['overlap_conflict']:.3f}")

print("\n✓ Interference metrics computed")
print_memory_usage()

In [ ]:
# Step 4: Pairwise merging and evaluation
print("\n[Step 4] Pairwise merging and evaluation...")
print("="*70)

# Load base state once
base_state_dict = torch.load('experts/base_state.pt')

pairwise_results = []

for i, (task1, task2) in enumerate(task_pairs):
    print(f"\n--- Pair {i+1}/{len(task_pairs)}: {task1.upper()}-{task2.upper()} ---")
    
    # TIES
    print("  TIES merging...")
    merged_ties = merge_ties_pair(base_state_dict, task1, task2, k_percent=CONFIG['sparsity_k'])
    scores_ties = evaluate_merged_on_pair(merged_ties, base_state_dict, task1, task2)
    print(f"    {task1.upper()}: {scores_ties[task1]:.4f}, {task2.upper()}: {scores_ties[task2]:.4f}")
    del merged_ties
    clear_memory()
    
    # UMTAM
    print("  UMTAM merging...")
    merged_umtam = merge_umtam_pair(base_state_dict, task1, task2, k_percent=CONFIG['sparsity_k'])
    scores_umtam = evaluate_merged_on_pair(merged_umtam, base_state_dict, task1, task2)
    print(f"    {task1.upper()}: {scores_umtam[task1]:.4f}, {task2.upper()}: {scores_umtam[task2]:.4f}")
    del merged_umtam
    clear_memory()
    
    improvement = scores_umtam['average'] - scores_ties['average']
    print(f"  UMTAM Δ: {improvement:+.4f}")
    
    pairwise_results.append({
        'task_pair': f"{task1}-{task2}",
        'task1': task1,
        'task2': task2,
        'ties_avg': scores_ties['average'],
        'umtam_avg': scores_umtam['average'],
        'ties_scores': {task1: scores_ties[task1], task2: scores_ties[task2]},
        'umtam_scores': {task1: scores_umtam[task1], task2: scores_umtam[task2]},
        'improvement': improvement,
    })
    
    print_memory_usage()

del base_state_dict
clear_memory()

print("\n✓ All pairwise merging complete")

---
## 📈 Analysis and Visualization

In [ ]:
import pandas as pd

# Combine data
combined_data = []
for inter, result in zip(interference_data, pairwise_results):
    combined_data.append({**inter, **result})

df = pd.DataFrame(combined_data)

# Summary table
print("\n" + "="*100)
print("TASK INTERFERENCE ANALYSIS RESULTS")
print("="*100)
print(f"{'Task Pair':<12} | {'Sign Conf':>10} | {'Sal Conf':>10} | {'Overlap':>10} | {'TIES':>8} | {'UMTAM':>8} | {'Δ':>10}")
print("-"*100)

for _, d in df.iterrows():
    print(f"{d['task_pair']:<12} | {d['sign_conflict_rate']:>10.3f} | {d['saliency_conflict']:>10.3f} | {d['overlap_conflict']:>10.3f} | {d['ties_avg']:>8.4f} | {d['umtam_avg']:>8.4f} | {d['improvement']:>+10.4f}")

print("="*100)

In [ ]:
# Correlation analysis
print("\n" + "="*70)
print("CORRELATION ANALYSIS")
print("="*70)

metrics_to_test = ['sign_conflict_rate', 'magnitude_conflict', 'saliency_conflict', 'overlap_conflict']
correlations = {}

for metric in metrics_to_test:
    x = df[metric].values
    y = df['improvement'].values
    
    if len(x) > 2:
        r, p = stats.pearsonr(x, y)
        correlations[metric] = {'r': r, 'p': p}
        sig = "**" if p < 0.05 else "*" if p < 0.1 else ""
        print(f"  {metric:<25}: r = {r:+.3f}, p = {p:.3f} {sig}")

print("\n  * p < 0.10, ** p < 0.05")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plot_metrics = [
    ('sign_conflict_rate', 'Sign Conflict Rate'),
    ('saliency_conflict', 'Saliency-Weighted Conflict'),
    ('overlap_conflict', 'Overlap Conflict (Top-k%)'),
    ('task_vector_distance', 'Task Vector Distance')
]

for ax, (metric, title) in zip(axes.flat, plot_metrics):
    x = df[metric].values
    y = df['improvement'].values * 100
    
    ax.scatter(x, y, s=150, c='#2E86AB', alpha=0.8, edgecolors='white', linewidths=2)
    
    for i, pair in enumerate(df['task_pair']):
        ax.annotate(pair.upper(), (x[i], y[i]), 
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=9, fontweight='bold')
    
    if len(x) > 2:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        r_val = correlations.get(metric, {}).get('r', 0)
        ax.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7, label=f'r={r_val:.2f}')
    
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    ax.set_xlabel(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('UMTAM Improvement (%)', fontsize=11, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Task Interference vs UMTAM Advantage', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('task_interference_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: task_interference_analysis.png")
plt.show()

In [ ]:
# Bar chart sorted by conflict
fig, ax = plt.subplots(figsize=(12, 6))

df_sorted = df.sort_values('sign_conflict_rate')
x = np.arange(len(df_sorted))
width = 0.35

bars1 = ax.bar(x - width/2, df_sorted['ties_avg'], width, label='TIES', color='#5DA5DA', alpha=0.8)
bars2 = ax.bar(x + width/2, df_sorted['umtam_avg'], width, label='UMTAM', color='#2E86AB', alpha=0.8)

for i, (_, row) in enumerate(df_sorted.iterrows()):
    improvement = row['improvement']
    color = 'green' if improvement > 0 else 'red'
    y_pos = max(row['ties_avg'], row['umtam_avg']) + 0.02
    ax.annotate(f"{improvement:+.3f}", (i, y_pos), ha='center', fontsize=10, fontweight='bold', color=color)

ax.set_xlabel('Task Pair (sorted by interference)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Score', fontsize=12, fontweight='bold')
ax.set_title('Pairwise Merging Performance', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([p.upper() for p in df_sorted['task_pair']], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('pairwise_merging_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: pairwise_merging_comparison.png")
plt.show()

In [ ]:
# Summary
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)

avg_improvement = df['improvement'].mean()
print(f"\nAverage UMTAM improvement: {avg_improvement:+.4f} ({avg_improvement*100:+.2f}%)")

# High vs Low conflict
median_conflict = df['sign_conflict_rate'].median()
high_conf = df[df['sign_conflict_rate'] >= median_conflict]
low_conf = df[df['sign_conflict_rate'] < median_conflict]

print(f"\nHigh-conflict pairs (>= {median_conflict:.3f}):")
print(f"  {', '.join(high_conf['task_pair'].str.upper())}")
print(f"  Avg improvement: {high_conf['improvement'].mean():+.4f}")

print(f"\nLow-conflict pairs (< {median_conflict:.3f}):")
print(f"  {', '.join(low_conf['task_pair'].str.upper())}")
print(f"  Avg improvement: {low_conf['improvement'].mean():+.4f}")

# Best correlation
if correlations:
    best = max(correlations.items(), key=lambda x: abs(x[1]['r']))
    print(f"\nStrongest correlation: {best[0]}")
    print(f"  r = {best[1]['r']:+.3f}, p = {best[1]['p']:.3f}")

print("="*80)

In [ ]:
# Save results
results = {
    'config': CONFIG,
    'interference_data': interference_data,
    'pairwise_results': pairwise_results,
    'correlations': {k: {'r': float(v['r']), 'p': float(v['p'])} for k, v in correlations.items()},
}

with open('task_interference_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✓ Saved: task_interference_results.json")

print("\nEXPERIMENT COMPLETE!")

In [ ]:
# LaTeX table
print("\n% LaTeX table for paper")
print(r"\begin{table}[H]")
print(r"\centering")
print(r"\caption{Task interference analysis showing correlation between parameter conflict and UMTAM advantage.}")
print(r"\label{tab:interference_analysis}")
print(r"\begin{tabular}{l|ccc|cc|r}")
print(r"\toprule")
print(r"\textbf{Pair} & \textbf{Sign} & \textbf{Sal.} & \textbf{Overlap} & \textbf{TIES} & \textbf{UMTAM} & \textbf{$\Delta$} \\")
print(r"\midrule")

for _, d in df.sort_values('sign_conflict_rate').iterrows():
    pair = d['task_pair'].upper().replace('-', '--')
    ties_str = f"{d['ties_avg']:.3f}"
    umtam_str = f"\\textbf{{{d['umtam_avg']:.3f}}}" if d['improvement'] > 0 else f"{d['umtam_avg']:.3f}"
    print(f"{pair} & {d['sign_conflict_rate']:.2f} & {d['saliency_conflict']:.2f} & {d['overlap_conflict']:.2f} & {ties_str} & {umtam_str} & {d['improvement']:+.3f} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")